# Stata Analysis Code Template

This is a template for Stata analysis code. It is designed to be used in a Jupyter notebook.

* Key rules: 
    - EVERY cell should be self-contained and executable, so all global settings should be repeated in every cell.
    - Each cell usually write one table, panel or figure.
    - Do not write custom `program define` wrappers, Mata code, or meta/diagnostic machinery (postfiles, model-metadata logging, coefficient dumps); call estimators directly and store results with `est store` (plus `estadd` for table rows).
    - Keep the codes as few lines as possible, minimize unnecessary code beyond shown in the examples below.

* Output rules (always follow):
    - Tables: export to four formats with the same basename — `.md`, `.rtf`, `.csv`, `.tex` — using the `esttab` options shown below (`cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label`; add `r2_p` for nonlinear models).
    - Figures: export to both `.pdf` and `.png`.
    - No extra side files (`.ster`, metadata, diagnostic dumps); the four formats are the deliverable.

* Entry file: one `reg.dta` file per sample (one for a project that uses only one baseline sample)

## Baseline regressions and sample descriptives


In [1]:
version 18
set more off
use "data/icw_loan_facilities/reg.dta", clear

gen double loan_size_usd_m=exp(loan_size) if !missing(loan_size)
label var loan_size_usd_m "Loan Size (USD millions)"
gen double assets_usd_m=exp(firm_size) if !missing(firm_size)
label var assets_usd_m "Firm Size (USD millions)"

global descvars "interest_rate financial_covenants general_covenants num_lenders loan_size_usd_m maturity pp_indicator secured assets_usd_m credit_rating profitability leverage tangibility"
est clear
estpost tabstat $descvars, stat(n mean sd p25 p50 p75) c(s)
esttab using "tables/descriptive_statistics.md", cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") replace label addnotes("Main complete-case facility population; PP Ratio uses the separate rated pre/post sample.")
esttab using "tables/descriptive_statistics.rtf", cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") replace label addnotes("Main complete-case facility population; PP Ratio uses the separate rated pre/post sample.")
esttab using "tables/descriptive_statistics.csv", cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") replace label addnotes("Main complete-case facility population; PP Ratio uses the separate rated pre/post sample.")
esttab using "tables/descriptive_statistics.tex", cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") replace label addnotes("Main complete-case facility population; PP Ratio uses the separate rated pre/post sample.")
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do

## Table 3: financial covenants


In [1]:
version 18
set more off
use "data/icw_loan_facilities/reg.dta", clear

global rhs "uncorrected corrected inst_investor revolver interest_rate secured loan_size maturity credit_rating num_lenders pp_indicator firm_size profitability leverage"
global fe "i.loan_year"
est clear
regress financial_covenants $rhs $fe, vce(cluster firmid)
tempvar clustertag
egen byte `clustertag'=tag(firmid) if e(sample)
quietly count if `clustertag'==1
local nclusters=r(N)
estadd scalar nclusters=`nclusters'
est store m1
esttab m1 using "tables/table3_financial_covenants.md", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label drop(*.loan_year) scalars("nclusters Borrower clusters") addnotes("Loan-year fixed effects included." "Standard errors clustered by borrower.")
esttab m1 using "tables/table3_financial_covenants.rtf", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label drop(*.loan_year) scalars("nclusters Borrower clusters") addnotes("Loan-year fixed effects included." "Standard errors clustered by borrower.")
esttab m1 using "tables/table3_financial_covenants.csv", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label drop(*.loan_year) scalars("nclusters Borrower clusters") addnotes("Loan-year fixed effects included." "Standard errors clustered by borrower.")
esttab m1 using "tables/table3_financial_covenants.tex", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label drop(*.loan_year) scalars("nclusters Borrower clusters") addnotes("Loan-year fixed effects included." "Standard errors clustered by borrower.")
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do

## Table 4: interest rate


In [1]:
version 18
set more off
use "data/icw_loan_facilities/reg.dta", clear

global rhs "uncorrected corrected inst_investor revolver financial_covenants loan_size maturity credit_rating num_lenders pp_indicator firm_size profitability leverage"
global fe "i.loan_year"
est clear
regress interest_rate $rhs $fe, vce(cluster firmid)
tempvar clustertag
egen byte `clustertag'=tag(firmid) if e(sample)
quietly count if `clustertag'==1
local nclusters=r(N)
estadd scalar nclusters=`nclusters'
est store m1
esttab m1 using "tables/table4_interest_rate.md", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label drop(*.loan_year) scalars("nclusters Borrower clusters") addnotes("Loan-year fixed effects included." "Standard errors clustered by borrower.")
esttab m1 using "tables/table4_interest_rate.rtf", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label drop(*.loan_year) scalars("nclusters Borrower clusters") addnotes("Loan-year fixed effects included." "Standard errors clustered by borrower.")
esttab m1 using "tables/table4_interest_rate.csv", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label drop(*.loan_year) scalars("nclusters Borrower clusters") addnotes("Loan-year fixed effects included." "Standard errors clustered by borrower.")
esttab m1 using "tables/table4_interest_rate.tex", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label drop(*.loan_year) scalars("nclusters Borrower clusters") addnotes("Loan-year fixed effects included." "Standard errors clustered by borrower.")
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do

## Table 5: collateral, general covenants, and maturity


In [1]:
version 18
set more off
use "data/icw_loan_facilities/reg.dta", clear

global fe "i.loan_year"
global cl "firmid"
est clear
probit secured uncorrected corrected inst_investor revolver loan_size maturity credit_rating num_lenders firm_size profitability leverage tangibility $fe, vce(cluster $cl)
tempvar clustertag1
egen byte `clustertag1'=tag($cl) if e(sample)
quietly count if `clustertag1'==1
local nclusters=r(N)
estadd scalar nclusters=`nclusters'
est store m_sec
regress general_covenants uncorrected corrected inst_investor revolver interest_rate secured loan_size maturity credit_rating num_lenders pp_indicator firm_size profitability leverage $fe, vce(cluster $cl)
tempvar clustertag2
egen byte `clustertag2'=tag($cl) if e(sample)
quietly count if `clustertag2'==1
local nclusters=r(N)
estadd scalar nclusters=`nclusters'
est store m_gen
regress ln_maturity uncorrected corrected inst_investor interest_rate financial_covenants secured loan_size credit_rating credit_rating_sq num_lenders pp_indicator firm_size profitability leverage tangibility $fe, vce(cluster $cl)
tempvar clustertag3
egen byte `clustertag3'=tag($cl) if e(sample)
quietly count if `clustertag3'==1
local nclusters=r(N)
estadd scalar nclusters=`nclusters'
est store m_mat
esttab m_sec m_gen m_mat using "tables/table5_loan_mechanisms.md", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label drop(*.loan_year) scalars("r2_p Pseudo R-squared" "nclusters Borrower clusters") mtitles("Secured" "General covenants" "Log maturity") addnotes("Loan-year fixed effects included." "Standard errors clustered by borrower.")
esttab m_sec m_gen m_mat using "tables/table5_loan_mechanisms.rtf", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label drop(*.loan_year) scalars("r2_p Pseudo R-squared" "nclusters Borrower clusters") mtitles("Secured" "General covenants" "Log maturity") addnotes("Loan-year fixed effects included." "Standard errors clustered by borrower.")
esttab m_sec m_gen m_mat using "tables/table5_loan_mechanisms.csv", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label drop(*.loan_year) scalars("r2_p Pseudo R-squared" "nclusters Borrower clusters") mtitles("Secured" "General covenants" "Log maturity") addnotes("Loan-year fixed effects included." "Standard errors clustered by borrower.")
esttab m_sec m_gen m_mat using "tables/table5_loan_mechanisms.tex", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label drop(*.loan_year) scalars("r2_p Pseudo R-squared" "nclusters Borrower clusters") mtitles("Secured" "General covenants" "Log maturity") addnotes("Loan-year fixed effects included." "Standard errors clustered by borrower.")
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do

## Table 6: performance-pricing ratio


In [1]:
version 18
set more off
use "data/icw_pp_ratio_eligible/reg.dta", clear

global fe "i.loan_year"
global cl "firmid"
est clear
probit pp_ratio uncorrected corrected inst_investor revolver secured loan_size maturity credit_rating num_lenders firm_size profitability leverage $fe, vce(cluster $cl)
tempvar clustertag
egen byte `clustertag'=tag($cl) if e(sample)
quietly count if `clustertag'==1
local nclusters=r(N)
estadd scalar nclusters=`nclusters'
est store m1
esttab m1 using "tables/table6_pp_ratio.md", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label drop(*.loan_year) scalars("r2_p Pseudo R-squared" "nclusters Borrower clusters") addnotes("Loan-year fixed effects included." "Standard errors clustered by borrower.")
esttab m1 using "tables/table6_pp_ratio.rtf", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label drop(*.loan_year) scalars("r2_p Pseudo R-squared" "nclusters Borrower clusters") addnotes("Loan-year fixed effects included." "Standard errors clustered by borrower.")
esttab m1 using "tables/table6_pp_ratio.csv", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label drop(*.loan_year) scalars("r2_p Pseudo R-squared" "nclusters Borrower clusters") addnotes("Loan-year fixed effects included." "Standard errors clustered by borrower.")
esttab m1 using "tables/table6_pp_ratio.tex", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label drop(*.loan_year) scalars("r2_p Pseudo R-squared" "nclusters Borrower clusters") addnotes("Loan-year fixed effects included." "Standard errors clustered by borrower.")
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do

In [1]:
version 18
set more off
use "data/icw_pp_ratio_eligible/reg.dta", clear

global descvars "pp_ratio"
est clear
estpost tabstat $descvars, stat(n mean sd p25 p50 p75) c(s)
esttab using "tables/descriptive_pp_ratio.md", cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") replace label addnotes("Rated PP facilities with classifiable pricing bases; borrowers have rated PP facilities before and after the first report.")
esttab using "tables/descriptive_pp_ratio.rtf", cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") replace label addnotes("Rated PP facilities with classifiable pricing bases; borrowers have rated PP facilities before and after the first report.")
esttab using "tables/descriptive_pp_ratio.csv", cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") replace label addnotes("Rated PP facilities with classifiable pricing bases; borrowers have rated PP facilities before and after the first report.")
esttab using "tables/descriptive_pp_ratio.tex", cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") replace label addnotes("Rated PP facilities with classifiable pricing bases; borrowers have rated PP facilities before and after the first report.")
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do